In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import re


In [5]:
df = pd.read_csv("netflix_titles.csv")

df.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [7]:
df = df[['title', 'director', 'cast', 'listed_in', 'description']]


In [9]:
df.fillna('', inplace=True)


In [10]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text


In [11]:
df['soup'] = (
    df['director'] + ' ' +
    df['cast'] + ' ' +
    df['listed_in'] + ' ' +
    df['description']
)

df['soup'] = df['soup'].apply(clean_text)


In [12]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['soup'])


In [13]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [14]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()


In [15]:
def recommend_movies(title, num_recommendations=5):
    if title not in indices:
        return "Movie not found in database."
    
    idx = indices[title]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:num_recommendations+1]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return df['title'].iloc[movie_indices]


In [16]:
recommend_movies("Inception")


3452      Peaky Blinders
6272    Before the Flood
6376               Brick
808       Sniper: Legacy
161        Mars Attacks!
Name: title, dtype: object

In [17]:

recommend_movies("The Matrix")


8415       The Matrix Reloaded
8416    The Matrix Revolutions
6501               Cloud Atlas
7151         Jupiter Ascending
636               The Ice Road
Name: title, dtype: object

In [18]:
recommend_movies("Ganglands")

2668                               Earth and Blood
3976                        The Eagle of El-Se'eed
3789                                Killer Ratings
4399                                       Warrior
7017    How to Live Mortgage Free with Sarah Beeny
Name: title, dtype: object

In [19]:
recommend_movies("Kota Factory")

8775         Yeh Meri Family
3466            Girls Hostel
2353           Chaman Bahaar
2472                  Betaal
266     The Creative Indians
Name: title, dtype: object

In [22]:
recommend_movies("Dhurandar")

'Movie not found in database.'